# Notebook 06 - Evaluate Language Adherence

Question: did the model actually answer in Kirundi?

This notebook uses AfroLID (`UBC-NLP/afrolid_1.5`), an African-language-aware language identification model. If the model cannot be downloaded or loaded, the notebook should fail visibly so the issue can be fixed.

In [ ]:
import os
import sys

from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())

sys.path.append(str(ROOT / 'src'))

from kirundi_sft_starter.evals import classify_language, load_response_table, summarize_language_adherence

from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml

load_env()
project_config = load_yaml(ROOT / 'configs/project.yaml')

## Load generated language responses

Notebook 05 writes one language-prompt response file per model under `results/responses/`. This notebook keeps a narrower job: load those files, run language identification, and save the comparison table.

In [ ]:
language_frames = []

for model_key, model_config in project_config['models']['registry'].items():
    response_path = ROOT / model_config['response_path']
    if response_path.exists():
        language_frames.append(load_response_table(response_path, model_key))

if not language_frames:
    raise FileNotFoundError('No response files found. Run the response generation cell in Notebook 05 first.')

language_responses = pd.concat(language_frames, ignore_index=True)

language_responses.head()

In [ ]:
from transformers import pipeline

language_id_pipeline = pipeline(
    'text-classification',
    model=project_config['evaluation']['language_id_model'],
    token=os.environ.get('HF_TOKEN'),
)

detected_languages = language_responses['response'].apply(lambda text: classify_language(text, language_id_pipeline)).apply(pd.Series)

language_scored = pd.concat([language_responses, detected_languages], axis=1)

language_summary = summarize_language_adherence(language_scored)

language_output_path = ROOT / project_config['evaluation']['language_results_path']

ensure_dir(language_output_path)

language_summary.to_csv(language_output_path, index=False)

print('Saved', language_output_path)

print(language_summary.to_string(index=False))

## Report table

| model | num_prompts | % Kirundi responses | notes |
|---|---:|---:|---|
| base | filled by notebook | filled by notebook | AfroLID |
| sft_raw | filled by notebook | filled by notebook | AfroLID |
| sft_adapted | filled by notebook | filled by notebook | AfroLID |